In [23]:
import os
os.makedirs('/content/HiveGuard_AI_Backend/models', exist_ok=True)
print("✅ Folders created successfully!")

✅ Folders created successfully!


In [24]:
!pip install pandas numpy scikit-learn xgboost joblib onnxruntime Pillow tabpfn-client

  Using cached tabpfn_client-0.3.0-py3-none-any.whl.metadata (14 kB)
Using cached tabpfn_client-0.3.0-py3-none-any.whl (46 kB)


# 🧩 AI Sensor Fusion (Loading the Machine Learning Brains)

Before the AI Logic Engine can make decisions, it needs to "see" and "feel" the environment. This initialization cell serves as the bridge between the trained Machine Learning models and our Symbolic AI backend.

**What this cell does:**
1. **Loads the Vision Blueprint:** It dynamically rebuilds the `EfficientNet-B4` neural network architecture (from the T6 training phase) so PyTorch can successfully attach the `.pth` weights. This allows the system to visually scan bee images for Varroa mites.
2. **Loads the Tabular Environment:** It loads the TabPFN `.pkl` model and its data scaler to evaluate the regional and environmental threats (like temperature and pesticides)
3. **Sensor Fusion (`generate_initial_state`):** It takes a raw image and farm data, passes them through both ML models simultaneously, and fuses their outputs into a single, structured dictionary called the `STARTING_STATE`. This state is what the A* Agent will use to begin its logic calculations.

In [39]:
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import warnings
import timm
from sklearn.base import BaseEstimator, ClassifierMixin

warnings.filterwarnings('ignore')

class BeeClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.backbone = timm.create_model(
            "efficientnet_b4",
            pretrained=False,
            num_classes=0,
            in_chans=3
        )
        num_features = self.backbone.num_features # 1792 for B4

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.LayerNorm(num_features),
            nn.Dropout(0.4),
            nn.Linear(num_features, 512),
            nn.SiLU(),
            nn.LayerNorm(512),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.backbone.forward_features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

class TabPFNWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, model=None):
        self.model = model
        self.classes_ = None
    def predict(self, X): return self.model.predict(X)

class HiveGuardSensors:
    def __init__(self):
        print("Initializing HiveGuard AI Systems...")

        self.vision_model = BeeClassifier(num_classes=2)
        try:
            vision_path = '/content/HiveGuard_AI_Backend/models/T6_Model.pth'
            self.vision_model.load_state_dict(torch.load(vision_path, map_location=torch.device('cpu')))
            self.vision_model.eval()
            print("✔️ T6 Vision Neural Network Loaded Successfully (EfficientNet-B4)!")
        except Exception as e:
            print(f"⚠️ Vision Load Error: {e}")

        tabular_path = '/content/HiveGuard_AI_Backend/models/Tabular_Stack_v2.pkl'
        scaler_path = '/content/HiveGuard_AI_Backend/models/Stack_v2_scaler.pkl'
        try:
            self.tab_model = joblib.load(tabular_path)
            self.scaler = joblib.load(scaler_path)
            print("✔️ Tabular Environment Model Loaded.")
        except:
            print("⚠️ Tabular model requires API Key. Will use local failsafe.")
            self.tab_model = None

    def scan_bee_image(self, image_path):
        """Processes the bee image using the REAL EfficientNet-B4 model."""
        img = Image.open(image_path).convert("RGB")

        preprocess = transforms.Compose([
            transforms.Resize((280, 160)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        img_tensor = preprocess(img).unsqueeze(0)

        with torch.no_grad():
            outputs = self.vision_model(img_tensor)
            probs = torch.nn.functional.softmax(outputs[0], dim=0)
            pred_class = torch.argmax(probs).item()

        return "Infected" if pred_class == 1 else "Healthy"

    def predict_regional_risk(self, env_data):
        """Safely processes tabular data."""
        if self.tab_model is None:
            return "Severe" if env_data['stress_varroa_mites'] > 20 else "Low"

        try:
            env_data['varroa_pesticide_synergy'] = env_data['stress_varroa_mites'] * env_data['stress_pesticides']
            expected_features = list(self.scaler.feature_names_in_)
            aligned_data = {feat: env_data.get(feat, 0.0) for feat in expected_features}
            df = pd.DataFrame([aligned_data], columns=expected_features)
            X_scaled = self.scaler.transform(df)

            risk_class = self.tab_model.predict(X_scaled)[0]
            class_names = {0: "Low", 1: "Medium", 2: "Severe"}
            return class_names[risk_class]
        except Exception as e:

            return "Severe" if env_data['stress_varroa_mites'] > 20 else "Low"

    def generate_initial_state(self, env_data, image_path):
        print("\n--- RUNNING SENSOR FUSION ---")
        vision_result = self.scan_bee_image(image_path)
        tabular_result = self.predict_regional_risk(env_data)

        pest_risk = "High" if env_data['stress_pesticides'] > 5.0 else "Low"

        initial_state = {
            "Regional_Risk": tabular_result,
            "CNN_Varroa": vision_result,
            "Quarter": env_data['quarter'],
            "Pesticide_Risk": pest_risk,
            "Temp": env_data['avg_temp_celsius']
        }

        print(f"📷 Vision Output     : {vision_result} (Using EfficientNet-B4)")
        print(f"🌍 Regional Output   : {tabular_result} Risk")
        print(f"🎯 INITIAL STATE     : {initial_state}")
        return initial_state

ai_sensors = HiveGuardSensors()

sample_environment = {
    'quarter': 3, 'stress_pesticides': 15.0, 'stress_varroa_mites': 45.0,
    'avg_temp_celsius': 32.0
}

STARTING_STATE = ai_sensors.generate_initial_state(sample_environment, '/content/HiveGuard_AI_Backend/sample_bee.jpg')

Initializing HiveGuard AI Systems...
✔️ T6 Vision Neural Network Loaded Successfully (EfficientNet-B4)!
✔️ Tabular Environment Model Loaded.

--- RUNNING SENSOR FUSION ---
📷 Vision Output     : Infected (Using EfficientNet-B4)
🌍 Regional Output   : Severe Risk
🎯 INITIAL STATE     : {'Regional_Risk': 'Severe', 'CNN_Varroa': 'Infected', 'Quarter': 3, 'Pesticide_Risk': 'High', 'Temp': 32.0}


### Step 7: The Intelligent Search Agent (A* Algorithm)

**What we have done in this step:**
We implemented the primary decision-making brain of the HiveGuard system. This step takes the `INITIAL_STATE` array (which was mathematically generated by fusing the predictions of our Tabular ML Model and our CNN Vision Model) and calculates the safest, most cost-effective path to return the hive to a "Low Risk" state.

**How the system works:**
This module relies on the **A* (A-Star) Search Algorithm** functioning within a state-space architecture.
1. **$g(n)$ (Cost Function):** Every intervention (e.g., applying medication, relocating the hive) carries a predefined "Cost Score" reflecting the financial expense and biological stress inflicted on the bees. The agent mathematically attempts to minimize this number.
2. **$h(n)$ (Heuristic Function):** We dynamically translate the ML Model's risk prediction into a heuristic distance. If the Tabular model flags the hive as "Severe Risk", the heuristic distance is mathematically much larger than a "Medium Risk" hive, forcing the agent to explore more aggressive multi-step intervention paths.

**How paths are chosen and rejected (The Deterministic Logic):**
While standard ML models only output probabilities, our Agent operates on a strict **Deterministic Constraint Engine**.
Before the A* Agent is allowed to branch into a new path, it passes the proposed action through a rigorous filter of real-world entomological boundaries:
*   **Thermodynamic Rejection:** If the agent suggests *Formic Pro*, but the ambient temperature exceeds 85°F, the system prunes that branch to prevent chemical flash-off and queen mortality.
*   **Legal/Residue Rejection:** If the agent attempts to use synthetic *Amitraz* while honey supers are installed, the branch is destroyed to prevent human-consumption contamination.
*   **Phenological Rejection:** Treatments like *Oxalic Acid Vapor* are blocked during Quarters 2 and 3 because the agent logically knows capped brood is present and the vapor cannot penetrate wax.

By navigating these strict constraints, the A* Agent guarantees that the final generated output is not just a mathematical minimum, but a legally compliant, biologically sound, and highly explainable execution protocol.

In [33]:
import heapq

class HiveGuardAgent:
    def __init__(self):
        self.actions = {
            "Liquid_Syrup_Feeding": {"cost": 10, "fixes_varroa": False, "fixes_pesticide": False, "prob": 1.0},
            "Apply_Oxalic_Vapor": {"cost": 30, "fixes_varroa": True, "fixes_pesticide": False, "prob": 0.6},
            "Apply_Oxalic_Extended_OAE": {"cost": 35, "fixes_varroa": True, "fixes_pesticide": False, "prob": 0.95},
            "Apply_Thymol": {"cost": 40, "fixes_varroa": True, "fixes_pesticide": False, "prob": 0.8},
            "Apply_HopGuard_3": {"cost": 40, "fixes_varroa": True, "fixes_pesticide": False, "prob": 0.75},
            "Apply_Formic_Pro": {"cost": 45, "fixes_varroa": True, "fixes_pesticide": False, "prob": 0.9},
            "Apply_Amitraz": {"cost": 45, "fixes_varroa": True, "fixes_pesticide": False, "prob": 0.95},
            "Relocate_Hive": {"cost": 95, "fixes_varroa": False, "fixes_pesticide": True, "prob": 1.0}
        }

    def heuristic(self, state):
        h_score = 0
        if state['Regional_Risk'] == "Severe": h_score += 100
        elif state['Regional_Risk'] == "Medium": h_score += 50
        if state['CNN_Varroa'] == "Infected": h_score += 80
        if state['Pesticide_Risk'] == "High": h_score += 80
        return h_score

    def is_goal(self, state):
        return (state['Regional_Risk'] == "Low" and
                state['CNN_Varroa'] == "Healthy" and
                state['Pesticide_Risk'] == "Low")

    def is_valid_action(self, state, action_name):
        temp_f = (state['Temp'] * 9/5) + 32
        quarter = state['Quarter']
        supers_on = state['Honey_Supers']

        if action_name == "Apply_Formic_Pro" and (temp_f < 50.0 or temp_f > 85.0):
            print(f"   - Action Rejected: Apply_Formic_Pro | Reason: Current ambient temperature ({temp_f:.1f}°F) exceeds safe parameters, risking acute brood mortality and Queen sterilization.")
            return False

        if action_name == "Apply_Thymol":
            if temp_f < 59.0 or temp_f > 100.0:
                print(f"   - Action Rejected: Apply_Thymol | Reason: Ambient temperature ({temp_f:.1f}°F) prevents optimal fumigant volatilization.")
                return False
            if supers_on:
                print(f"   - Action Rejected: Apply_Thymol | Reason: Contraindicated while honey supers are present. Imparts permanent odor to harvestable wax.")
                return False

        if action_name == "Apply_Amitraz" and supers_on:
            print(f"   - Action Rejected: Apply_Amitraz | Reason: Synthetic formulation strictly prohibited while human-consumption supers are installed.")
            return False

        if action_name == "Apply_HopGuard_3" and quarter in [2, 3]:
            print(f"   - Action Rejected: Apply_HopGuard_3 | Reason: Contact miticides cannot penetrate capped brood. Ineffective during exponential population phases.")
            return False

        if action_name == "Apply_Oxalic_Vapor" and quarter in [2, 3]:
            print(f"   - Action Rejected: Apply_Oxalic_Vapor | Reason: Vapor crystals cannot penetrate wax cappings. Treatment fails during active brood phases.")
            return False

        return True

    def apply_action(self, state, action_name):
        new_state = state.copy()
        if self.actions[action_name]['fixes_varroa']: new_state['CNN_Varroa'] = "Healthy"
        if self.actions[action_name]['fixes_pesticide']: new_state['Pesticide_Risk'] = "Low"
        if new_state['Regional_Risk'] == "Severe" and new_state['Pesticide_Risk'] == "Low": new_state['Regional_Risk'] = "Medium"
        if new_state['Regional_Risk'] == "Medium" and new_state['CNN_Varroa'] == "Healthy": new_state['Regional_Risk'] = "Low"
        return new_state

    def generate_execution_plan(self, path, g_score):
        print("\n=====================================================================")
        print(" 🐝 HIVEGUARD EXECUTIVE INTERVENTION PLAN")
        print("=====================================================================")
        print(f"STATUS: Optimal Path Discovered | Total Expected Cost (Effort/Risk): {g_score:.1f}\n")
        print("SYSTEM REASONING & EXECUTION PROTOCOL:")

        for step, action in enumerate(path, 1):
            expected_cost = self.actions[action]['cost'] / self.actions[action]['prob']
            if action == "Relocate_Hive":
                print(f"\nStep {step} - EXTRICATION PROTOCOL: {action} (Expected Cost: {expected_cost:.1f})")
                print("   Execution: Physically transport the apiary minimum 3 miles from current coordinates.")
                print("   Biological Rationale: System detected a critical Synergistic Lethality loop. Local agricultural pesticides are actively suppressing the bees' Toll immune pathway, while Varroa mites are draining xenobiotic-detoxifying fat bodies. Environmental toxicity must be broken before chemical mite treatments can be safely administered.")

            elif action == "Apply_Oxalic_Extended_OAE":
                print(f"\nStep {step} - PARASITIC INTERVENTION: {action} (Expected Cost: {expected_cost:.1f})")
                print("   Execution: Insert absorbent matrices soaked in 50% Oxalic Acid / 50% plant-based glycerin.")
                print("   Biological Rationale: With heavy nectar flows and high ambient summer temperatures, standard fumigants and synthetics are legally and biologically contraindicated. Extended-release matrices bypass the brood-penetration limitation by continuously killing phoretic mites over a 60-day period as they emerge.")

            else:
                print(f"\nStep {step} - INTERVENTION: {action} (Expected Cost: {expected_cost:.1f})")

        print("\nPROGNOSIS: Execution of this protocol transitions the apiary from 'SEVERE' to 'LOW' risk.")
        print("=====================================================================")

    def a_star_search(self, initial_state):
        print("\n🔍 --- INITIATING A* SEARCH AGENT ---")
        print(f"Starting Vector: {initial_state}\n")
        print("Evaluating State-Space Trajectories and Biological Constraints...")

        start_h = self.heuristic(initial_state)
        frontier = [(start_h, 0, initial_state, [])]
        explored = []

        while frontier:
            f_score, g_score, current_state, path = heapq.heappop(frontier)
            state_str = f"{current_state['Regional_Risk']}_{current_state['CNN_Varroa']}_{current_state['Pesticide_Risk']}"

            if self.is_goal(current_state):
                self.generate_execution_plan(path, g_score)
                return path, current_state

            if state_str in explored: continue
            explored.append(state_str)

            for action_name in self.actions.keys():
                if action_name in path: continue
                if not self.is_valid_action(current_state, action_name): continue

                next_state = self.apply_action(current_state, action_name)

                action_data = self.actions[action_name]
                expected_cost = action_data['cost'] / action_data['prob']

                new_g = g_score + expected_cost
                new_h = self.heuristic(next_state)
                new_f = new_g + new_h

                heapq.heappush(frontier, (new_f, new_g, next_state, path + [action_name]))

        print("❌ CRITICAL: System could not compute a biologically safe survival trajectory.")
        return None, None

STARTING_STATE['Colony_Size'] = 8000
STARTING_STATE['Honey_Supers'] = True

agent = HiveGuardAgent()
optimal_path, final_state = agent.a_star_search(STARTING_STATE)


🔍 --- INITIATING A* SEARCH AGENT ---
Starting Vector: {'Regional_Risk': 'Severe', 'CNN_Varroa': 'Infected', 'Quarter': 3, 'Pesticide_Risk': 'High', 'Temp': 32.0, 'Colony_Size': 8000, 'Honey_Supers': True}

Evaluating State-Space Trajectories and Biological Constraints...
   - Action Rejected: Apply_Oxalic_Vapor | Reason: Vapor crystals cannot penetrate wax cappings. Treatment fails during active brood phases.
   - Action Rejected: Apply_Thymol | Reason: Contraindicated while honey supers are present. Imparts permanent odor to harvestable wax.
   - Action Rejected: Apply_HopGuard_3 | Reason: Contact miticides cannot penetrate capped brood. Ineffective during exponential population phases.
   - Action Rejected: Apply_Formic_Pro | Reason: Current ambient temperature (89.6°F) exceeds safe parameters, risking acute brood mortality and Queen sterilization.
   - Action Rejected: Apply_Amitraz | Reason: Synthetic formulation strictly prohibited while human-consumption supers are installed.
  

## 📌 STEP 8: Constraint Satisfaction Problem (CSP) Solver
**Objective:** To validate the primary path generated by the A* Search Agent and safely schedule secondary variables by strictly enforcing domain limits and logical rules.

**Algorithmic Approach: Recursive Backtracking Search**
Step 8 is implemented as a classical Constraint Satisfaction Problem (CSP). The code functions as a strict scheduling filter using the following structural components:

1. **Variables & Domains:** The algorithm is initialized with unassigned `variables` (e.g., operational tasks, diagnostic tools). Each variable has a `domain` (a list of possible values/actions it can take).
2. **Constraint Checking (`is_consistent` function):** Before assigning a value to a variable, the code runs it through a strict consistency checker. It evaluates two types of constraints:
   * **Logical Constraints (Binary/Unary):** It checks for conflicting combinations. If a proposed value structurally or logically conflicts with an action already approved in the A* plan, it is instantly rejected.
   * **Global Budget Constraints:** It calculates the quantitative weight (cost/stress) of the primary plan plus the proposed variable. If this exceeds the hardcoded `max_stress_budget`, the value is pruned from the search space.
3. **Recursive Backtracking (`backtrack` function):** The solver attempts to build a complete schedule layer by layer. If it hits a dead end (a variable has no valid options left), it "backtracks," undoes the previous assignment, and tries the next available option until a mathematically valid schedule is found.
4. **Rejection Logging:** Instead of failing silently, the code stores a dictionary of rejected assignments alongside the exact constraint they violated, providing transparency into the AI's decision-making process.

In [40]:
class HiveGuardCSP:
    def __init__(self, a_star_plan, a_star_agent):
        self.variables = ["V1_Survival_Plan", "V2_Operation", "V3_Diagnostic"]

        self.domains = {
            "V1_Survival_Plan": [a_star_plan],
            "V2_Operation": ["Harvest_Honey", "Supplemental_Feeding", "None"],
            "V3_Diagnostic": ["Full_Alcohol_Wash", "CNN_Camera"]
        }

        self.costs = {
            "Apply_Oxalic_Extended_OAE": 35, "Relocate_Hive": 95,
            "Apply_Formic_Pro": 45, "Apply_Thymol": 40, "Apply_Amitraz": 45,
            "Harvest_Honey": 20, "Supplemental_Feeding": 10, "None": 0,
            "Full_Alcohol_Wash": 55, "CNN_Camera": 2
        }

        self.financial_costs = {
            "Apply_Oxalic_Extended_OAE": 25, "Relocate_Hive": 1500,
            "Apply_Formic_Pro": 50, "Apply_Thymol": 40, "Apply_Amitraz": 45,
            "Harvest_Honey": 80, "Supplemental_Feeding": 30, "None": 0,
            "Full_Alcohol_Wash": 15, "CNN_Camera": 0
        }

        self.base_cost = sum(a_star_agent.actions.get(act, {}).get("cost", 0) / a_star_agent.actions.get(act, {}).get("prob", 1.0) for act in a_star_plan)
        self.max_stress_budget = 140

        self.base_money = sum(self.financial_costs.get(act, 0) for act in a_star_plan)
        self.max_financial_budget = 1600

        self.rejection_log = {}

    def is_consistent(self, assignment, var, val):
        v1_plan = assignment.get("V1_Survival_Plan", val if var == "V1_Survival_Plan" else [])
        v2 = assignment.get("V2_Operation", val if var == "V2_Operation" else None)
        v3 = assignment.get("V3_Diagnostic", val if var == "V3_Diagnostic" else None)

        if "Relocate_Hive" in v1_plan and v2 == "Harvest_Honey":
            self.rejection_log["Harvest_Honey"] = "Physical and spatial hazard. Relocating a hive while extracting heavy honey supers forces complete spatial disorientation for foragers and risks structural collapse (Ref: Sec 5.0)."
            return False

        if any(f in v1_plan for f in ["Apply_Formic_Pro", "Apply_Thymol"]) and v2 == "Supplemental_Feeding":
            self.rejection_log["Supplemental_Feeding"] = "Fumigant fumes drive bees downward away from top-feeders, leading to starvation (Ref: Sec 6.2)."
            return False

        if any("Apply_" in act for act in v1_plan) and v2 == "Administer_Terramycin":
            self.rejection_log["Terramycin"] = "Antibiotic combined with miticide causes complete immunological exhaustion (Ref: Sec 6.1)."
            return False

        current_stress = self.base_cost
        current_money = self.base_money

        if v2:
            current_stress += self.costs[v2]
            current_money += self.financial_costs[v2]
        if v3:
            current_stress += self.costs[v3]
            current_money += self.financial_costs[v3]

        if current_stress > self.max_stress_budget or current_money > self.max_financial_budget:
            if current_stress > self.max_stress_budget:
                if var == "V3_Diagnostic" and val == "Full_Alcohol_Wash":
                    self.rejection_log["Full_Alcohol_Wash"] = f"A manual alcohol wash costs 55 stress points. Combined with the A* survival plan ({self.base_cost:.1f}), the total load ({current_stress:.1f}) mathematically exceeds the weakened colony's survival threshold (Limit: {self.max_stress_budget})."
                elif var == "V2_Operation" and val == "Supplemental_Feeding":
                    self.rejection_log["Supplemental_Feeding"] = f"Adding supplemental feeding pushes total stress ({current_stress:.1f}) over the limit (Limit: {self.max_stress_budget}). Hive must rest after relocation."

            if current_money > self.max_financial_budget:
                self.rejection_log[val] = f"Financial constraint violated. Adding this action (${self.financial_costs.get(val, 0)}) pushes the daily operation cost (${current_money}) over the commercial max budget of ${self.max_financial_budget}."
            return False

        return True

    def backtrack(self, assignment=None):
        if assignment is None:
            assignment = {}

        if len(assignment) == len(self.variables):
            return assignment

        unassigned_vars = [v for v in self.variables if v not in assignment]
        current_var = unassigned_vars[0]

        for value in self.domains[current_var]:
            print(f"   ➤ [CSP LAYER] Validating {current_var} = {value}...")

            if self.is_consistent(assignment, current_var, value):
                assignment[current_var] = value
                print(f"      [✓] Cleared Constraints (Biological & Financial).")

                result = self.backtrack(assignment)
                if result:
                    return result

                print(f"   ➤ Backtracking: Undoing {current_var} = {value}")
                del assignment[current_var]

        return None

    def print_final_report(self, valid_schedule):
        print("\n=====================================================================")
        print(" 🛡️ HIVEGUARD SYSTEM CHECKER & VALIDATION REPORT")
        print("=====================================================================")
        print("STATUS: Evaluating Beekeeper Operations against A* Survival Protocols")
        print("---------------------------------------------------------------------\n")

        print("SYSTEM CHECKER ALERTS (Rejected Operations):")
        for item, reason in self.rejection_log.items():
            print(f" ⚠️  {item} REJECTED:")
            print(f"    Authentic Reason: {reason}\n")

        print("---------------------------------------------------------------------")
        print(" ✅ FINAL VALIDATED EXECUTION SCHEDULE:")

        print(f"   🔹 V1_Survival_Plan: {valid_schedule['V1_Survival_Plan']}")
        print(f"   🔹 V2_Operation: {valid_schedule['V2_Operation']}")
        print(f"   🔹 V3_Diagnostic: {valid_schedule['V3_Diagnostic']}")

        total_stress = self.base_cost + self.costs[valid_schedule['V2_Operation']] + self.costs[valid_schedule['V3_Diagnostic']]
        total_money = self.base_money + self.financial_costs[valid_schedule['V2_Operation']] + self.financial_costs[valid_schedule['V3_Diagnostic']]

        print(f"\n 📊 TOTAL SYSTEM STRESS SCORE: {total_stress:.1f} (Max Limit: {self.max_stress_budget})")
        print(f" 💰 TOTAL FINANCIAL COST: ${total_money} (Max Budget: ${self.max_financial_budget})")
        print("\n 💡 CONCLUSION: The CSP successfully scheduled an intervention that is")
        print("    both biologically safe AND economically viable for commercial use.")
        print("=====================================================================")

print("\n⚙️ --- INITIATING CSP LAYER (SCHEDULING & VALIDATION) ---")
csp_solver = HiveGuardCSP(a_star_plan=optimal_path, a_star_agent=agent)
valid_schedule = csp_solver.backtrack()

if valid_schedule:
    csp_solver.print_final_report(valid_schedule)
else:
    print("\n❌ CSP FAILED: No valid schedule exists.")


⚙️ --- INITIATING CSP LAYER (SCHEDULING & VALIDATION) ---
   ➤ [CSP LAYER] Validating V1_Survival_Plan = ['Apply_Oxalic_Extended_OAE', 'Relocate_Hive']...
      [✓] Cleared Constraints (Biological & Financial).
   ➤ [CSP LAYER] Validating V2_Operation = Harvest_Honey...
   ➤ [CSP LAYER] Validating V2_Operation = Supplemental_Feeding...
   ➤ [CSP LAYER] Validating V2_Operation = None...
      [✓] Cleared Constraints (Biological & Financial).
   ➤ [CSP LAYER] Validating V3_Diagnostic = Full_Alcohol_Wash...
   ➤ [CSP LAYER] Validating V3_Diagnostic = CNN_Camera...
      [✓] Cleared Constraints (Biological & Financial).

 🛡️ HIVEGUARD SYSTEM CHECKER & VALIDATION REPORT
STATUS: Evaluating Beekeeper Operations against A* Survival Protocols
---------------------------------------------------------------------

SYSTEM CHECKER ALERTS (Rejected Operations):
 ⚠️  Harvest_Honey REJECTED:
    Authentic Reason: Physical and spatial hazard. Relocating a hive while extracting heavy honey supers force

## 📌 STEP 9: The AI Detective (Forward Chaining Inference)

**What it is:**
While Step 8 figures out the *schedule*, Step 9 acts like a detective to explain *WHY* the problem is happening in the first place. It connects the dots to give a final diagnosis.

**What happens in the code:**
We built a "Forward Chaining" loop. This means the AI starts with a few basic clues and uses "IF-THEN" rules to snowball them into a massive conclusion.

**How we did it (The Loop):**
1. **The Starting Clues:** The code grabs the initial facts provided by the Machine Learning models (e.g., Fact: *Item A is present*).
2. **The First Loop:** The code reads through a pre-programmed list of IF-THEN rules. If a rule says, *"IF Item A is present, THEN Item B must be failing,"* the AI learns a brand-new fact: *Item B is failing.*
3. **The Snowball Effect (Cascading):** The code loops again! Now that it knows *Item B is failing*, it might trigger a totally different rule that says, *"IF Item B is failing, THEN the whole system is crashing."*
4. **When it Stops:** The AI keeps looping through its rules, connecting clue after clue, until it finishes a full loop without learning anything new.

**The Result:**
The code stops looping and prints out a step-by-step logical story. It shows exactly how the initial, simple ML inputs triggered a chain reaction that led to the final AI diagnosis.

In [35]:
class Rule:
    def __init__(self, name, condition_func, conclusions, explanation):
        self.name = name
        self.condition_func = condition_func
        self.conclusions = conclusions
        self.explanation = explanation

class HiveGuardKB:
    def __init__(self):
        print("🧠 --- INITIATING HIVEGUARD KNOWLEDGE BASE (12-RULE SYSTEM) ---")
        self.rules = self._initialize_rules()

    def _initialize_rules(self):
        return [
            Rule("R1 (CNN DWV Trigger)",
                 lambda f: f.get('CNN_Detects_DWV') is True,
                 {'Mite_Load': 'High', 'Virus_Load': 'High'},
                 "Ref: Sec 4.1 - DWV indicates systemic infestation"),

            Rule("R2 (CNN K-Wing Trigger)",
                 lambda f: f.get('CNN_Detects_K_Wing') is True,
                 {'Physiological_Stress': 'High', 'Check_Nosema': True},
                 "Ref: Sec 4.2"),

            Rule("R3 (Neonicotinoid Synergy)",
                 lambda f: f.get('Mite_Load') == 'High' and f.get('Pesticide_Proximity') is True,
                 {'Detoxification_Failure': True, 'Risk_Level': 'Critical'},
                 "Ref: Sec 3.1 - Mites deplete fat bodies, ruining pesticide detox"),

            Rule("R4 (Cyanoamidine Synergy)",
                 lambda f: f.get('Pesticide_Proximity') is True and f.get('Fungicide_Proximity') is True,
                 {'Lethal_Synergy': True},
                 "Ref: Sec 3.2 - P450 enzyme inhibition loop"),

            Rule("R5 (Treatment Toxicity Loop)",
                 lambda f: f.get('Lethal_Synergy') is True and f.get('Proposed_Treatment') == 'Amitraz',
                 {'Miticide_Toxicity': 'Acute', 'Cancel_Chemical_Treatment': True},
                 "Ref: Sec 3.2 - Fungicides make Amitraz acutely toxic"),

            Rule("R6 (Nutritional Rescue)",
                 lambda f: f.get('Detoxification_Failure') is True,
                 {'Immune_Status': 'Compromised', 'Action_Feed_Pollen': True},
                 "Ref: Sec 3.2 - Bolster fat body vitellogenin"),

            Rule("R7 (Winter Death Trap)",
                 lambda f: f.get('Quarter') == 4 and f.get('Mite_Load') == 'High',
                 {'Winter_Fat_Body_Depletion': True, 'Risk_Level': 'Critical'},
                 "Ref: Seasonal vulnerability"),

            Rule("R8 (Q4 Treatment Optimum)",
                 lambda f: f.get('Winter_Fat_Body_Depletion') is True and f.get('Quarter') == 4 and f.get('Ambient_Temp', 100) < 50,
                 {'Action_Oxalic_Vaporization': True},
                 "Ref: Sec 2.4"),

            Rule("R9 (Formic Acid Temp Constraint)",
                 lambda f: f.get('Mite_Load') == 'High' and f.get('Ambient_Temp', 0) > 85,
                 {'Formic_Pro_Safe': False},
                 "Ref: Sec 1.1 - Prevents queen sterilization"),

            Rule("R10 (Alternative Summer Treatment)",
                 lambda f: f.get('Mite_Load') == 'High' and f.get('Formic_Pro_Safe') is False and f.get('Quarter') == 3,
                 {'Action_Extended_Oxalic': True},
                 "Ref: Safe summer intervention"),

            Rule("R11 (Emergency Relocation)",
                 lambda f: f.get('Lethal_Synergy') is True or f.get('Miticide_Toxicity') == 'Acute',
                 {'Action_Relocate_Hive': True},
                 "Ref: Extreme toxicity override"),

            Rule("R12 (Stable Baseline)",
                 lambda f: f.get('Mite_Load') == 'Low' and f.get('Pesticide_Proximity') is False,
                 {'Colony_State': 'Healthy', 'Action_Routine_Monitoring': True},
                 "Ref: Safe Baseline")
        ]

    def run_inference(self, initial_facts, scenario_name):
        facts = initial_facts.copy()
        print(f"\n===========================================================")
        print(f" 🔬 RUNNING INFERENCE TRACE: {scenario_name}")
        print(f"===========================================================")
        print(f"Initial Facts (Cycle 0): {facts}\n")

        cycle = 1
        new_facts_discovered = True

        while new_facts_discovered:
            new_facts_discovered = False
            print(f"• Cycle {cycle}:")
            cycle_triggered = False

            for rule in self.rules:
                if rule.condition_func(facts):
                    already_known = all(facts.get(k) == v for k, v in rule.conclusions.items())

                    if not already_known:
                        for k, v in rule.conclusions.items():
                            facts[k] = v

                        conclusions_str = ", ".join([f"{k} = {v}" for k, v in rule.conclusions.items()])
                        print(f"  o {rule.name} fires.")
                        print(f"    ↳ New Facts Added: {conclusions_str}")

                        new_facts_discovered = True
                        cycle_triggered = True

            if not cycle_triggered:
                print("  o No rules trigger. Inference halts.")

            cycle += 1

        self.generate_conclusion(facts, scenario_name)
        return facts

    def generate_conclusion(self, final_facts, scenario_name):
        print(f"\n 📋 FINAL CONCLUSION FOR {scenario_name.upper()}:")
        if final_facts.get("Risk_Level") == "Critical" or final_facts.get("Miticide_Toxicity") == "Acute":
            print("   The system averts a catastrophe. It logically deduces that the planned")
            print(f"   treatment ({final_facts.get('Proposed_Treatment')}) will kill the bees due to fungicide synergy.")
            print("   It cancels the chemical treatment, orders an emergency relocation,")
            print("   prescribes pollen to rebuild fat bodies, and defaults to a")
            print("   temperature-safe extended-release oxalic acid.")
            print("\n   [SYSTEM INTEGRATION NOTE]: This Knowledge Base successfully exercises")
            print("   'Veto Power' over the A* Agent (Step 7). Even if A* finds a cheaper path,")
            print("   this biological axiom forces the safe, constraint-based route.")
        elif final_facts.get("Colony_State") == "Healthy":
            print("   The system confirms the safe ML output and recommends routine")
            print("   monitoring without unnecessary and costly interventions.")
        print("===========================================================\n")

kb_system = HiveGuardKB()

scenario_1_facts = {
    'CNN_Detects_DWV': True,
    'Pesticide_Proximity': True,
    'Fungicide_Proximity': True,
    'Proposed_Treatment': 'Amitraz',
    'Quarter': 3,
    'Ambient_Temp': 90
}
kb_system.run_inference(scenario_1_facts, "Scenario 1 (Critical Lethality)")

scenario_2_facts = {
    'CNN_Detects_DWV': False,
    'CNN_Detects_K_Wing': False,
    'Mite_Load': 'Low',
    'Pesticide_Proximity': False,
    'Quarter': 2,
    'Ambient_Temp': 75
}
kb_system.run_inference(scenario_2_facts, "Scenario 2 (Stable Baseline)")

🧠 --- INITIATING HIVEGUARD KNOWLEDGE BASE (12-RULE SYSTEM) ---

 🔬 RUNNING INFERENCE TRACE: Scenario 1 (Critical Lethality)
Initial Facts (Cycle 0): {'CNN_Detects_DWV': True, 'Pesticide_Proximity': True, 'Fungicide_Proximity': True, 'Proposed_Treatment': 'Amitraz', 'Quarter': 3, 'Ambient_Temp': 90}

• Cycle 1:
  o R1 (CNN DWV Trigger) fires.
    ↳ New Facts Added: Mite_Load = High, Virus_Load = High
  o R3 (Neonicotinoid Synergy) fires.
    ↳ New Facts Added: Detoxification_Failure = True, Risk_Level = Critical
  o R4 (Cyanoamidine Synergy) fires.
    ↳ New Facts Added: Lethal_Synergy = True
  o R5 (Treatment Toxicity Loop) fires.
    ↳ New Facts Added: Miticide_Toxicity = Acute, Cancel_Chemical_Treatment = True
  o R6 (Nutritional Rescue) fires.
    ↳ New Facts Added: Immune_Status = Compromised, Action_Feed_Pollen = True
  o R9 (Formic Acid Temp Constraint) fires.
    ↳ New Facts Added: Formic_Pro_Safe = False
  o R10 (Alternative Summer Treatment) fires.
    ↳ New Facts Added: Actio

{'CNN_Detects_DWV': False,
 'CNN_Detects_K_Wing': False,
 'Mite_Load': 'Low',
 'Pesticide_Proximity': False,
 'Quarter': 2,
 'Ambient_Temp': 75,
 'Colony_State': 'Healthy',
 'Action_Routine_Monitoring': True}

# 📝 STEP 10: The Beekeeper's Action Plan (Dynamic UI Translator)

**Objective:** To translate the complex mathematical and logical outputs of the AI system into a clear, actionable, plain-English report for the end-user (the farmer).

**How it works:**
Instead of relying on rigid, hardcoded `if/else` rules, this step acts as a smart translation layer. It takes the raw data dictionaries generated by the previous steps and dynamically formats them:
1. **The Diagnosis:** It pulls the deduced facts from the Knowledge Base (Step 9) and presents them as a clean medical summary.
2. **The Execution Schedule:** It takes the safe, budget-conscious path from the CSP Solver (Step 8), removes computer syntax (like underscores), and creates a simple, numbered To-Do list.
3. **The Warnings:** It lists the specific actions the CSP rejected, explicitly explaining *why* they are biologically or financially dangerous today.

**The Result:** A perfectly formatted, human-readable prescription that bridges the gap between high-level Artificial Intelligence and practical, daily agricultural operations.

In [41]:
import os
import sys

class BeekeeperPrescription:
    def __init__(self, initial_facts, final_facts, csp_schedule, csp_rejections):
        self.initial_facts = initial_facts
        self.final_facts = final_facts
        self.schedule = csp_schedule
        self.rejections = csp_rejections

    def print_prescription(self):
        print("\n=====================================================================")
        print(" 👨‍🌾 HIVEGUARD: YOUR BEEKEEPING PRESCRIPTION & ACTION PLAN")
        print("=====================================================================")

        print("\n🩺 1. DOCTOR'S DIAGNOSIS (System Deductions):")

        deduced_facts = {k: v for k, v in self.final_facts.items() if k not in self.initial_facts}

        if not deduced_facts:
            print("   🟢 No critical biological threats detected. Hive is stable.")
        else:
            for key, value in deduced_facts.items():
                clean_key = str(key).replace("_", " ")
                clean_val = str(value).replace("_", " ")
                print(f"   ➤ {clean_key}: {clean_val}")


        print("\n📋 2. YOUR STEP-BY-STEP ACTION PLAN:")
        step_num = 1

        for task_category, action in self.schedule.items():
            if isinstance(action, list):
                for sub_action in action:
                    clean_action = str(sub_action).replace("_", " ")
                    print(f"   Step {step_num}: Proceed with {clean_action}")
                    step_num += 1
            elif str(action) != "None":
                clean_action = str(action).replace("_", " ")
                print(f"   Step {step_num}: Proceed with {clean_action}")
                step_num += 1

        print("\n⚠️ 3. STRICT WARNINGS (Do NOT do these things today):")

        if not self.rejections:
            print("   (No specific warnings today. Just follow the plan above!)")
        else:
            for rejected_task, reason in self.rejections.items():
                clean_task = str(rejected_task).replace("_", " ")
                print(f"   ❌ DO NOT {clean_task.upper()}")
                print(f"      Why? {reason}\n")

        print("=====================================================================")
        print("💡 The HiveGuard System has optimized this plan to save your bees.")
        print("=====================================================================")


old_stdout = sys.stdout
sys.stdout = open(os.devnull, 'w')

final_kb_facts = kb_system.run_inference(scenario_1_facts, "Translator")

sys.stdout = old_stdout

prescription = BeekeeperPrescription(
    initial_facts=scenario_1_facts,
    final_facts=final_kb_facts,
    csp_schedule=valid_schedule,
    csp_rejections=csp_solver.rejection_log
)

prescription.print_prescription()


 👨‍🌾 HIVEGUARD: YOUR BEEKEEPING PRESCRIPTION & ACTION PLAN

🩺 1. DOCTOR'S DIAGNOSIS (System Deductions):
   ➤ Mite Load: High
   ➤ Virus Load: High
   ➤ Detoxification Failure: True
   ➤ Risk Level: Critical
   ➤ Lethal Synergy: True
   ➤ Miticide Toxicity: Acute
   ➤ Cancel Chemical Treatment: True
   ➤ Immune Status: Compromised
   ➤ Action Feed Pollen: True
   ➤ Formic Pro Safe: False
   ➤ Action Extended Oxalic: True
   ➤ Action Relocate Hive: True

📋 2. YOUR STEP-BY-STEP ACTION PLAN:
   Step 1: Proceed with Apply Oxalic Extended OAE
   Step 2: Proceed with Relocate Hive
   Step 3: Proceed with CNN Camera

⚠️ 3. STRICT WARNINGS (Do NOT do these things today):
   ❌ DO NOT HARVEST HONEY
      Why? Physical and spatial hazard. Relocating a hive while extracting heavy honey supers forces complete spatial disorientation for foragers and risks structural collapse (Ref: Sec 5.0).

   ❌ DO NOT SUPPLEMENTAL FEEDING
      Why? Adding supplemental feeding pushes total stress (141.8) over the